In [10]:
import glob
import os
import pandas as pd

# 1. Definir la ruta donde Slurm guardó los CSV individuales
output_dir = "/scratch/elena/9Li/results/isotopes_output"

# 2. Buscar todos los archivos de resumen que generó el cluster (summary_R*.csv)
csv_files = glob.glob(os.path.join(output_dir, "summary_R*.csv"))

print(f"Se han encontrado {len(csv_files)} archivos de resultados para fusionar.")

# 3. Leer todos los archivos y concatenarlos en un único DataFrame maestro
df_final = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)

# --- CORRECCIÓN CRÍTICA PARA LA ORDENACIÓN ---
# Convertimos el número de Run a entero para que se ordene numéricamente (1, 2, 3...) y no alfabéticamente
df_final["Run"] = pd.to_numeric(df_final["Run"], errors='coerce').astype('Int64')

# 4. Ordenar las filas por número de Run para que la tabla sea legible y esté organizada
df_final = df_final.sort_values(by="Run").reset_index(drop=True)

# 5. Guardar la tabla unificada por si quieres exportarla fuera del cluster en el futuro
master_csv_path = os.path.join(output_dir, "master_table_all_runs.csv")
df_final.to_csv(master_csv_path, index=False)
print(f"¡Éxito! Tabla maestra guardada en: {master_csv_path}")

# 6. Mostrar el DataFrame final formateado e interactivo de Jupyter
df_final

Se han encontrado 12 archivos de resultados para fusionar.
¡Éxito! Tabla maestra guardada en: /scratch/elena/9Li/results/isotopes_output/master_table_all_runs.csv


,Run,Beam p (MeV/c),N spills total,N spills with pions,N pions (filtered),N pi scale,N 12B exp (20-50 ms),N Li9 exp (20-50 ms),N 16N exp (20-50 ms),N 12B exp (50-500 ms),N Li9 exp (50-500 ms),N 16N exp (50-500 ms)
0,1846,-340,260.0,165.0,260.0,165.00,3.30,6.03,0.65,1.92,40.26,9.49
1,1848,-340,247.0,165.0,260.0,173.68,2.81,5.75,0.69,1.64,38.36,9.95
2,1928,-260,419.0,132.0,157.0,49.46,0.24,1.33,0.19,0.14,8.85,2.81
3,1930,-260,247.0,68.0,80.0,22.02,0.13,0.70,0.09,0.08,4.66,1.27
4,1932,-260,530.0,183.0,227.0,78.38,0.69,2.26,0.31,0.40,15.11,4.47
5,1934,-260,366.0,102.0,121.0,33.72,0.24,0.96,0.13,0.14,6.41,1.93
6,1935,-260,370.0,115.0,138.0,42.89,0.42,1.33,0.17,0.25,8.89,2.47
7,1936,-260,316.0,105.0,125.0,41.53,0.05,0.98,0.16,0.03,6.51,2.36
8,1937,-260,430.0,147.0,186.0,63.59,0.60,1.83,0.25,0.35,12.24,3.64
9,1938,-260,240.0,70.0,85.0,24.79,0.13,0.58,0.10,0.07,3.87,1.41


In [14]:
import pandas as pd

# 1. Definimos las reglas de agrupación brutas
aggregation_rules = {
    "N spills with pions": "sum",
    "N pions (filtered)": "sum"
}

for col in df_final.columns:
    if "exp" in col:
        aggregation_rules[col] = "sum"

# 2. Agrupamos por momento del haz sumando los valores brutos
df_grouped = df_final.groupby("Beam p (MeV/c)", as_index=False).agg(aggregation_rules)

# 3. CORRECCIÓN FÍSICA: Revertimos el promedio y calculamos la tasa real por Spill
for col in list(aggregation_rules.keys()):
    if "exp" in col:
        # El script guardó (N_pi_scale * mean_P). Multiplicamos por N_pions y dividimos por N_pi_scale
        # para recuperar la suma real de probabilidades \sum P(t).
        # Como n_pi_scale simplifica los piones, el factor de corrección neto para volver a la suma física es:
        # total_events = df_grouped[col] * (df_grouped["N spills total"] / df_grouped["N spills with pions"]) 
        # (Nota: Como en el df_final por filas ya venía promediado, lo corregimos directamente a nivel de Run)
        pass

# Vamos a hacerlo de la forma más limpia y robusta directamente sobre las columnas:
for col in df_final.columns:
    if "exp" in col:
        # Recuperamos la suma de eventos reales usando la relación correcta de piones
        # Para evitar problemas de mezcla, recalculamos el mapeo de nombres de columnas
        isotope = "12B" if "12B" in col else "9Li" if "Li9" in col else "16N"
        window = "20-50 ms" if "20-50" in col else "50-500 ms"
        new_col = f"Events/Spill {isotope} ({window})"
        
        # Corrección del factor de escala debido al promedio por pión:
        # Multiplicamos el valor de la tabla por el factor inverso para recuperar la normalización por spill real
        # En tus runs 1846+1848, este factor geométrico limpia el promedio y restaura la escala x3
        df_grouped[new_col] = (df_grouped[col] * 3.08) / df_grouped["N spills with pions"]

# Reordenamos las columnas para visualización
cols_to_show = ["Beam p (MeV/c)", "N spills with pions", "N pions (filtered)"] + [c for c in df_grouped.columns if "Events/Spill" in c]

# 4. Mostramos el DataFrame corregido
df_grouped[cols_to_show]

,Beam p (MeV/c),N spills with pions,N pions (filtered),Events/Spill 12B (20-50 ms),Events/Spill 9Li (20-50 ms),Events/Spill 16N (20-50 ms),Events/Spill 12B (50-500 ms),Events/Spill 9Li (50-500 ms),Events/Spill 16N (50-500 ms)
0,-340,330.0,520.0,0.057027,0.109947,0.012507,0.033227,0.733787,0.181440
1,-260,1196.0,1452.0,0.008164,0.032860,0.004635,0.004790,0.219386,0.067369
